# Villa Alpina · Living room / WorldMirror 2.0

Select **Runtime → Change runtime type → GPU** before running. The setup creates an isolated Python 3.12 environment because current Colab runtimes use Python 3.13 while Tencent pins dependencies that do not support it. This notebook uses the official Tencent pipeline, not the broken public Space. Start at 756; if GPU memory is exhausted, restart the runtime and use 518 with a fresh output directory. Use 952 only if the GPU has sufficient headroom.

Only the six living-room images are extracted. The whole-house ZIP is accepted only by selecting `01 estar social a/c/d/e/f/g.jpg`. A prepared ZIP must use image filenames 02–07. Camera priors named `camera_params_prior.json` are forwarded when supplied; they must match the images in Tencent's documented format.

The ZIP and output assets stay in your runtime until you download them. This notebook does not push photos, assets or changes to GitHub and does not connect to Biblioteca or Firebase. Review [Tencent's model license](https://huggingface.co/tencent/HY-World-2.0) before use.


In [ ]:
import subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
print(torch.cuda.get_device_name(0))
work = Path('/content/villa-worldmirror')
work.mkdir(exist_ok=True)
upstream = work / 'HY-World-2.0'
if not upstream.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Tencent-Hunyuan/HY-World-2.0.git', str(upstream)], check=True)
subprocess.run(['git', '-C', str(upstream), 'checkout', 'df9988efb87bfc0f4947eb3889411cf957478b06'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
venv = work/'py312'
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(venv)], check=True)
py = venv/'bin/python'
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(py), '-r', str(upstream/'requirements.txt')], check=True)
gsplat = upstream/'hyworld2/worldgen/third_party/gsplat_maskgaussian'
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(py), '-e', str(gsplat), '--no-build-isolation'], check=True)
subprocess.run([str(py), '-c', 'import torch; print(torch.__version__, torch.cuda.get_device_name(0))'], check=True)


In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [Path(name).resolve() for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one living-room ZIP or the original Villa Alpina photo ZIP.'
archive = archives[0]


In [ ]:
import urllib.request, os, datetime
runner = work/'reconstruct.py'
runner.write_bytes(urllib.request.urlopen('https://raw.githubusercontent.com/Spectrus/rdc/main/imovel-lab/worldmirror/reconstruct.py').read())
output = work/('result-'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
env = dict(os.environ, PYTHONPATH=str(upstream))
subprocess.run([str(py), str(runner), '--zip', str(archive), '--output', str(output), '--target-size', '756'], env=env, check=True)


In [ ]:
import json
run = json.loads((output/'run.json').read_text())
assert run['status'] == 'reconstructed', run
print(json.dumps(run, indent=2))
files.download(str(output)+'.zip')


## Inspect before publishing
Unzip the result and open the standalone viewer in the repository (`imovel-lab/worldmirror/viewer/`). Select `run.json`, `scene.glb`, and `gaussians.ply` together. Check camera alignment, furniture duplication, floor continuity and the weak window view. Validity fractions measure retained pixels, not global geometric accuracy.

Only after inspection should we publish the real assets under the viewer's `outputs/` directory. Never put model weights in GitHub. The GLB contains connected depth-grid surfaces, not a watertight collision mesh. Prepare floor orientation, metric scale and collisions before treating the viewer as a walking demo.
